# 16 — E3 Shuffled-Embedding Control (góp ý 6 của thầy)

**Mục tiêu:** kiểm tra lợi ích của RLMRec-Con đến từ *ngữ nghĩa đúng đối tượng* hay chỉ từ *loss alignment phụ*.

Embedding LLM (`usr_emb_np.pkl`, `itm_emb_np.pkl`) được hoán vị theo hàng: user *u* nhận embedding của một user khác, tương tự với sách.
Mọi thứ khác giữ nguyên như E1 RLMRec-Con: commit RLMRec, `lightgcn_plus.yml` (kd_weight 0.01, kd_temperature 0.2, 3 layer, …), seeds 2023/2024/2025, early stopping theo Recall@20 validation, Top-K chỉ mask train.

| Chế độ | User embedding | Item embedding |
|---|---|---|
| `user_shuffle` | xáo | đúng |
| `item_shuffle` | đúng | xáo |
| `both_shuffle` | xáo | xáo |

9 run (3 chế độ × 3 seeds), mỗi run khoảng 9 phút trên T4. Thêm 1 run kiểm tra (`none`, seed 2023, không xáo) để chắc pipeline tái lập đúng E1.

**Cách chạy:** Runtime → GPU (T4) → *Run all*. Nếu Colab bị ngắt: chạy lại *Run all*, các run đã xong được bỏ qua.

**Đọc kết quả:** nếu xáo làm kết quả tụt về gần LightGCN → lợi ích đến từ ngữ nghĩa đúng đối tượng; nếu vẫn gần RLMRec-Con → lợi ích chủ yếu đến từ loss phụ (regularization).

In [ ]:
# CELL 1 — CẤU HÌNH
import os, sys, json, time, shutil, pickle, subprocess, math
from pathlib import Path
import numpy as np
import torch

MODES = ['user_shuffle', 'item_shuffle', 'both_shuffle']
SEEDS = [2023, 2024, 2025]
PERM_SEED = 14            # seed cố định cho hoán vị (dùng chung cho mọi run)
RUN_SANITY = True         # 1 run không xáo (seed 2023) để đối chiếu với E1 RLMRec-Con

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
ROOT = Path(os.environ.get('SNA_ROOT', '/content/drive/MyDrive/Data_SNA_Group14'))
DATA_DIR = ROOT / 'data/raw/amazon'
OUT_ROOT = ROOT / 'experiments/E3/shuffle_control'
WORK = Path(os.environ.get('SNA_WORK', '/content/shuffle_work'))   # đĩa local Colab, chép lên Drive khi run xong
REPO = WORK / 'RLMRec'
RLMREC_URL = 'https://github.com/HKUDS/RLMRec.git'
RLMREC_COMMIT = '22413752246de3dee8ab0d509f7f7a8889080f95'

for f in ['trn_mat.pkl', 'val_mat.pkl', 'tst_mat.pkl', 'usr_emb_np.pkl', 'itm_emb_np.pkl']:
    assert (DATA_DIR / f).is_file(), f'Missing {DATA_DIR / f}'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('Output:', OUT_ROOT)

In [ ]:
# CELL 2 — CLONE RLMRec ĐÚNG COMMIT + LINK DATA
if torch.cuda.is_available():
    if not REPO.exists():
        subprocess.run(['git', 'clone', '-q', RLMREC_URL, str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '--detach', RLMREC_COMMIT], check=True)
    assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == RLMREC_COMMIT
    assert not subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip(), 'RLMRec repo bị sửa'
    target = REPO / 'data/amazon'
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists() and not target.is_symlink():
        target.symlink_to(DATA_DIR, target_is_directory=True)
    assert target.resolve() == DATA_DIR.resolve()
    print('RLMRec', RLMREC_COMMIT[:10], '| data ->', target.resolve())
else:
    print('Không có GPU: bỏ qua clone (chỉ chạy được các cell tổng hợp).')

In [ ]:
# CELL 3 — TẠO HOÁN VỊ (cố định, lưu lên Drive) + KIỂM TRA ĐÃ PHÁ ĐƯỢC SỰ KHỚP NGỮ NGHĨA
with open(DATA_DIR / 'usr_emb_np.pkl', 'rb') as f: usr_emb = np.asarray(pickle.load(f))
with open(DATA_DIR / 'itm_emb_np.pkl', 'rb') as f: itm_emb = np.asarray(pickle.load(f))
with open(DATA_DIR / 'trn_mat.pkl', 'rb') as f: trn = pickle.load(f).tocsr()
n_users, n_items = trn.shape
assert usr_emb.shape[0] == n_users and itm_emb.shape[0] == n_items

PERM_FILE = OUT_ROOT / 'permutations.npz'
if PERM_FILE.exists():
    z = np.load(PERM_FILE)
    user_perm, item_perm = z['user_perm'], z['item_perm']
    assert int(z['perm_seed']) == PERM_SEED, 'permutations.npz trên Drive dùng seed khác'
    print('Dùng lại hoán vị đã lưu:', PERM_FILE)
else:
    rng = np.random.default_rng(PERM_SEED)
    user_perm = rng.permutation(n_users)
    item_perm = rng.permutation(n_items)
    np.savez(PERM_FILE, user_perm=user_perm, item_perm=item_perm, perm_seed=PERM_SEED)
    print('Đã lưu hoán vị:', PERM_FILE)
assert sorted(user_perm.tolist()) == list(range(n_users)) and sorted(item_perm.tolist()) == list(range(n_items))
print('Số user giữ nguyên vị trí:', int((user_perm == np.arange(n_users)).sum()),
      '| số sách giữ nguyên vị trí:', int((item_perm == np.arange(n_items)).sum()))

# Mức khớp ngữ nghĩa: cosine trung bình giữa embedding user và embedding sách user đã tương tác (train).
def unit(a):
    a = a.astype(np.float32)
    return a / np.linalg.norm(a, axis=1, keepdims=True)
U, I = unit(usr_emb), unit(itm_emb)
rows, cols = trn.nonzero()
rand_cols = np.random.default_rng(0).integers(0, n_items, len(cols))
def mean_cos(Um, Im, r, c):
    total = sum(float(np.einsum('ij,ij->i', Um[r[s:s+20000]], Im[c[s:s+20000]]).sum()) for s in range(0, len(r), 20000))
    return total / len(r)
align = {
    'none (đúng)': mean_cos(U, I, rows, cols),
    'user_shuffle': mean_cos(U[user_perm], I, rows, cols),
    'item_shuffle': mean_cos(U, I[item_perm], rows, cols),
    'both_shuffle': mean_cos(U[user_perm], I[item_perm], rows, cols),
    'cặp ngẫu nhiên (tham chiếu)': mean_cos(U, I, rows, rand_cols),
}
for k, v in align.items():
    print(f'  cosine(user, sách train) {k:<28} {v:.4f}')
(OUT_ROOT / 'semantic_alignment_check.json').write_text(json.dumps(align, indent=2, ensure_ascii=False), encoding='utf-8')
del U, I

In [ ]:
# CELL 4 — SCRIPT CHẠY 1 RUN (mỗi run 1 process riêng vì RLMRec nạp config lúc import)
RUNNER = WORK / 'shuffle_runner.py'
RUNNER.write_text(r"""
import argparse, json, os, sys, pickle, traceback
from copy import deepcopy
from datetime import datetime, timezone
import numpy as np
import torch

ap = argparse.ArgumentParser()
ap.add_argument('--repo'); ap.add_argument('--data'); ap.add_argument('--perm')
ap.add_argument('--mode'); ap.add_argument('--seed', type=int); ap.add_argument('--out')
ap.add_argument('--commit')
a = ap.parse_args()
assert a.mode in ('none', 'user_shuffle', 'item_shuffle', 'both_shuffle')
os.makedirs(a.out, exist_ok=True)
os.chdir(a.repo)
sys.path.insert(0, os.path.join(a.repo, 'encoder'))
sys.argv = ['train_encoder.py', '--model', 'lightgcn_plus', '--dataset', 'amazon', '--cuda', '0', '--seed', str(a.seed)]
from config.configurator import configs
from trainer.trainer import init_seed
from data_utils.build_data_handler import build_data_handler
from models.bulid_model import build_model
from trainer.logger import Logger
from trainer.build_trainer import build_trainer

# --- thay embedding ngữ nghĩa bằng bản đã xáo (chỉ thay đổi duy nhất so với E1) ---
perm = np.load(a.perm)
u0, i0 = configs['usrprf_embeds'], configs['itmprf_embeds']
if a.mode in ('user_shuffle', 'both_shuffle'):
    configs['usrprf_embeds'] = u0[perm['user_perm']]
if a.mode in ('item_shuffle', 'both_shuffle'):
    configs['itmprf_embeds'] = i0[perm['item_perm']]
assert configs['usrprf_embeds'].shape == u0.shape and configs['itmprf_embeds'].shape == i0.shape
changed = dict(user=bool((configs['usrprf_embeds'] != u0).any()), item=bool((configs['itmprf_embeds'] != i0).any()))
assert changed['user'] == (a.mode in ('user_shuffle', 'both_shuffle'))
assert changed['item'] == (a.mode in ('item_shuffle', 'both_shuffle'))

configs['train']['seed'] = a.seed
configs['train']['save_model'] = False   # tự lưu best checkpoint vào thư mục run
assert configs['model']['name'] == 'lightgcn_plus' and 20 in configs['test']['k']
K_VALUES = configs['test']['k']

init_seed()
handler = build_data_handler(); handler.load_data()
model = build_model(handler).to(configs['device'])
trainer = build_trainer(handler, Logger())

history, best, captured, n_epochs = [], dict(epoch=None, recall20=-1.0), {}, [0]
o_eval, o_test, o_epoch = trainer.evaluate, trainer.test, trainer.train_epoch
fmt = lambda r: {f'{m}@{k}': float(v) for m, vs in r.items() for k, v in zip(K_VALUES, vs)}
def t_eval(m, epoch_idx=None):
    r = o_eval(m, epoch_idx)
    if epoch_idx is not None:
        v = fmt(r); history.append(dict(epoch_index=int(epoch_idx), **v))
        if v['recall@20'] > best['recall20']: best.update(epoch=int(epoch_idx), recall20=v['recall@20'])
    return r
def t_test(m):
    r = o_test(m); captured['metrics'] = fmt(r); captured['model'] = m
    return r
def t_epoch(m, e):
    o_epoch(m, e); n_epochs[0] += 1
trainer.evaluate, trainer.test, trainer.train_epoch = t_eval, t_test, t_epoch

trainer.train(model)
best_model = captured['model']
torch.save(best_model.state_dict(), os.path.join(a.out, 'best_model.pth'))

# --- Top-20 chỉ mask train (giống E1/E3A export) ---
with open(os.path.join(a.data, 'trn_mat.pkl'), 'rb') as f: trn = pickle.load(f).tocsr()
with open(os.path.join(a.data, 'tst_mat.pkl'), 'rb') as f: tst = pickle.load(f).tocsr()
users = np.flatnonzero(np.diff(tst.indptr) > 0).astype(np.int64)
assert len(users) == 9860
best_model.eval(); best_model.is_training = False; best_model.final_embeds = None
items_out, scores_out = [], []
with torch.no_grad():
    uv, iv = best_model.forward(best_model.adj, 1.0)
    for s in range(0, len(users), 256):
        part = users[s:s + 256]
        sc = uv[torch.as_tensor(part, device=uv.device)] @ iv.T
        rr, cc = [], []
        for j, u in enumerate(part):
            seen = trn.indices[trn.indptr[u]:trn.indptr[u + 1]]
            rr += [j] * len(seen); cc += seen.tolist()
        sc[torch.as_tensor(rr, device=sc.device, dtype=torch.long), torch.as_tensor(cc, device=sc.device, dtype=torch.long)] = -torch.inf
        ts, ti = torch.topk(sc, 20, dim=1)
        assert torch.isfinite(ts).all()
        items_out.append(ti.cpu().numpy().astype(np.int32)); scores_out.append(ts.cpu().numpy().astype(np.float32))
items = np.concatenate(items_out); scores = np.concatenate(scores_out)
recomputed = {}
disc = 1 / np.log2(np.arange(2, 22))
for k in (10, 20):
    rec, nd = [], []
    for u, row in zip(users, items):
        pos = tst.indices[tst.indptr[u]:tst.indptr[u + 1]]
        h = np.isin(row[:k], pos)
        rec.append(h.sum() / len(pos)); nd.append((h * disc[:k]).sum() / disc[:min(k, len(pos))].sum())
    recomputed[f'recall@{k}'] = float(np.mean(rec)); recomputed[f'ndcg@{k}'] = float(np.mean(nd))
diff = max(abs(recomputed[k] - captured['metrics'][k]) for k in recomputed)
assert diff < 1e-5, f'Top-K metrics lệch trainer: {diff}'
os.makedirs(os.path.join(a.out, 'topk'), exist_ok=True)
np.savez_compressed(os.path.join(a.out, 'topk', 'topk_test.npz'), user_ids=users, item_ids=items, scores=scores)

json.dump(history, open(os.path.join(a.out, 'validation_history.json'), 'w'), indent=2)
json.dump(dict(experiment='E3_Shuffle_Control', mode=a.mode, seed=a.seed, model='lightgcn_plus',
               rlmrec_commit=a.commit, embeddings_changed=changed,
               perm_seed=int(perm['perm_seed']), kd_weight=float(configs['model']['kd_weight']),
               epochs_completed=n_epochs[0], best_epoch_index=best['epoch'],
               best_validation_recall20=best['recall20'], test_metrics=captured['metrics'],
               topk_recomputed=recomputed, topk_max_abs_diff=diff, ranking_mask='train_only',
               finished_utc=datetime.now(timezone.utc).isoformat(), status='completed'),
          open(os.path.join(a.out, 'metrics.json'), 'w'), indent=2)
print('RUN DONE', a.mode, a.seed, captured['metrics'])
""", encoding='utf-8')
print('Runner:', RUNNER)

In [ ]:
# CELL 5 — CHẠY CÁC RUN (bỏ qua run đã xong; run hỏng thì chạy lại từ đầu)
def run_done(d):
    m = d / 'metrics.json'
    return m.is_file() and (d / 'topk/topk_test.npz').is_file() and json.loads(m.read_text()).get('status') == 'completed'

jobs = ([('none', 2023)] if RUN_SANITY else []) + [(m, s) for m in MODES for s in SEEDS]
for mode, seed in jobs:
    dst = OUT_ROOT / mode / f'seed_{seed}'
    if run_done(dst):
        print(f'[skip] {mode} seed {seed} đã xong'); continue
    assert torch.cuda.is_available(), 'Cần GPU để train'
    local = WORK / 'runs' / mode / f'seed_{seed}'
    shutil.rmtree(local, ignore_errors=True); local.mkdir(parents=True)
    t0 = time.time()
    print(f'[run ] {mode} seed {seed} ...', flush=True)
    with open(local / 'training_console.log', 'w') as log:
        p = subprocess.run([sys.executable, str(RUNNER), '--repo', str(REPO), '--data', str(DATA_DIR),
                            '--perm', str(OUT_ROOT / 'permutations.npz'), '--mode', mode, '--seed', str(seed),
                            '--out', str(local), '--commit', RLMREC_COMMIT],
                           stdout=log, stderr=subprocess.STDOUT)
    tail = (local / 'training_console.log').read_text(errors='ignore').splitlines()[-15:]
    if p.returncode != 0:
        print('\n'.join(tail)); raise RuntimeError(f'Run lỗi: {mode} seed {seed}')
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(local, dst)
    assert run_done(dst)
    met = json.loads((dst / 'metrics.json').read_text())
    print(f'[done] {mode} seed {seed} | {(time.time() - t0) / 60:.1f} phút | epochs {met["epochs_completed"]} '
          f'| Recall@20 {met["test_metrics"]["recall@20"]:.4f} NDCG@20 {met["test_metrics"]["ndcg@20"]:.4f}', flush=True)
print('ALL RUNS DONE')

In [ ]:
# CELL 6 — KIỂM TRA PIPELINE: run không xáo phải gần E1 RLMRec-Con seed 2023
sanity = OUT_ROOT / 'none/seed_2023/metrics.json'
e1 = ROOT / 'experiments/E1/lightgcn_plus/seed_2023/metrics.json'
if sanity.is_file() and e1.is_file():
    a = json.loads(sanity.read_text())['test_metrics']; b = json.loads(e1.read_text())['test_metrics']
    for k in ['recall@10', 'recall@20', 'ndcg@10', 'ndcg@20']:
        print(f'{k:<10} pipeline mới {a[k]:.5f} | E1 {b[k]:.5f} | lệch {a[k] - b[k]:+.5f}')
    print('Lệch nhỏ (< ~0.002) là do GPU không hoàn toàn tất định; lệch lớn thì dừng lại kiểm tra.')
else:
    print('Thiếu file để đối chiếu:', sanity if not sanity.is_file() else e1)

In [ ]:
# CELL 7 — TỔNG HỢP: cùng cách tính với E2 (từ Top-K), mean ± SD (ddof=0) qua 3 seeds
import pandas as pd
from scipy.stats import spearmanr

with open(DATA_DIR / 'tst_mat.pkl', 'rb') as f: tst = pickle.load(f).tocsr()
deg = np.asarray(trn.sum(0)).ravel()
n_head = math.ceil(0.2 * n_items)
head = np.zeros(n_items, bool); head[np.lexsort((np.arange(n_items), -deg))[:n_head]] = True
tail = ~head

EXP = ROOT / 'experiments'
def first(pattern):
    hits = sorted(EXP.glob(pattern)); return hits[0] if hits else None
SOURCES = {'LightGCN': {s: EXP / f'predictions/lightgcn/seed_{s}/topk_test.npz' for s in SEEDS},
           'RLMRec-Con': {s: EXP / f'predictions/lightgcn_plus/seed_{s}/topk_test.npz' for s in SEEDS},
           'Align user-only': {s: first(f'E3/runs/user_only/*/seed_{s}/topk/topk_test.npz') for s in SEEDS},
           'Align item-only': {s: first(f'E3/runs/item_only/*/seed_{s}/topk/topk_test.npz') for s in SEEDS}}
for m in MODES:
    SOURCES['Shuffle ' + m.replace('_shuffle', '')] = {s: OUT_ROOT / m / f'seed_{s}/topk/topk_test.npz' for s in SEEDS}

def metrics(path, K=20):
    z = np.load(path, allow_pickle=False)
    users, items = z['user_ids'], z['item_ids'][:, :K]
    hit = np.asarray(tst[users[:, None], items].todense()).astype(bool)
    ngt = np.diff(tst.indptr)[users]
    disc = 1 / np.log2(np.arange(2, K + 2)); idcg = np.r_[0, np.cumsum(disc)][np.minimum(ngt, K)]
    gt = tst[users]; hgt = np.asarray(gt[:, head].sum(1)).ravel(); tgt = np.asarray(gt[:, tail].sum(1)).ravel()
    hh = (hit & head[items]).sum(1); th = (hit & tail[items]).sum(1)
    expo = np.bincount(items.ravel(), minlength=n_items)
    return {'Recall@20': (hit.sum(1) / ngt).mean(), 'NDCG@20': ((hit * disc).sum(1) / idcg).mean(),
            'ARP': deg[items].mean(), 'APLT': tail[items].mean(),
            'HeadRecall': (hh[hgt > 0] / hgt[hgt > 0]).mean(), 'TailRecall': (th[tgt > 0] / tgt[tgt > 0]).mean(),
            'Coverage': (expo > 0).mean(), 'Spearman': spearmanr(deg, expo)[0]}

rows = []
for model, runs in SOURCES.items():
    for s, p in runs.items():
        if p is None or not Path(p).is_file():
            print('thiếu', model, s); continue
        rows.append(dict(model=model, seed=s, **metrics(p)))
per_run = pd.DataFrame(rows)
per_run.to_csv(OUT_ROOT / 'shuffle_per_run.csv', index=False)

cols = ['Recall@20', 'NDCG@20', 'ARP', 'APLT', 'HeadRecall', 'TailRecall', 'Coverage', 'Spearman']
g = per_run.groupby('model', sort=False)[cols]
summary = g.mean().add_suffix('_mean').join(g.std(ddof=0).add_suffix('_sd')).join(g.size().rename('n_seeds'))
summary.to_csv(OUT_ROOT / 'shuffle_summary.csv')
view = pd.DataFrame({c: [f"{summary.loc[m, c + '_mean']:.4f} ± {summary.loc[m, c + '_sd']:.4f}" for m in summary.index]
                     for c in cols}, index=summary.index)
view.insert(0, 'n_seeds', summary['n_seeds'])
display(view)

In [ ]:
# CELL 8 — SO VỚI RLMRec-Con THEO TỪNG SEED + PHẦN CẢI THIỆN CÒN GIỮ LẠI
# retained = (Shuffle - LightGCN) / (RLMRec-Con - LightGCN): ~1 = xáo không ảnh hưởng, ~0 = mất hết cải thiện
piv = per_run.pivot(index='seed', columns='model')
out = []
for model in [m for m in summary.index if m.startswith('Shuffle') or m.startswith('Align')]:
    for c in ['Recall@20', 'NDCG@20']:
        if model not in piv[c]: continue
        d = piv[c][model] - piv[c]['RLMRec-Con']
        gain = piv[c]['RLMRec-Con'] - piv[c]['LightGCN']
        out.append(dict(model=model, metric=c,
                        delta_vs_Con_mean=d.mean(), delta_vs_Con_min=d.min(), delta_vs_Con_max=d.max(),
                        seeds_below_Con=int((d < 0).sum()),
                        retained_gain=float(((piv[c][model] - piv[c]['LightGCN']) / gain).mean())))
comp = pd.DataFrame(out)
comp.to_csv(OUT_ROOT / 'shuffle_vs_con.csv', index=False)
display(comp.round(4))

In [ ]:
# CELL 9 — HÌNH
import matplotlib.pyplot as plt
order = [m for m in ['LightGCN', 'Shuffle both', 'Shuffle user', 'Shuffle item', 'RLMRec-Con'] if m in summary.index]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, c in zip(axes, ['Recall@20', 'NDCG@20']):
    ax.bar(range(len(order)), summary.loc[order, c + '_mean'], yerr=summary.loc[order, c + '_sd'], capsize=4, color='#888')
    for s in SEEDS:
        ax.scatter(range(len(order)), [piv[c][m].get(s, np.nan) for m in order], s=12, color='k', zorder=3)
    lo = summary.loc[order, c + '_mean'].min(); hi = summary.loc[order, c + '_mean'].max()
    ax.set_ylim(lo - 2 * (hi - lo + 1e-4), hi + (hi - lo + 1e-4))
    ax.set_xticks(range(len(order)), order, rotation=15); ax.set_title(c); ax.grid(axis='y', alpha=.3, ls='--')
fig.suptitle('E3 — Shuffled-embedding control (3 seeds)')
fig.tight_layout()
fig.savefig(OUT_ROOT / 'shuffle_control.png', dpi=200, bbox_inches='tight')
fig.savefig(OUT_ROOT / 'shuffle_control.pdf', bbox_inches='tight')
plt.show()
print('Kết quả lưu tại:', OUT_ROOT)